# 24 — Analysis: inputs, requirement checks, figures and poster numbers (Milestone 3)

Owner: Member 4. This notebook only **reads** the files that `15`, `20`–`23` write (Drive
`MyDrive/arxiv-project/`, never committed). It does not run the pipeline and depends on no other notebook, so it
works while some files are still missing: they are listed as `missing`, with who writes them.

| Section | Functions |
|---|---|
| 0. Setup | settings, paths |
| 1. Inputs | `ARTIFACTS`, `load_m3_artifacts`, `read_manual_sheets`, `read_manual_old`, `print_inputs` |
| 2. Statement requirements | `labels_for_check`, `check_requirements`, `print_checks` |
| 3. Run | `run_analysis` |
| 4. Tests | `AnalysisTests` |

Figures (labels, features, MRR, ablation) are added in the next step.

**The numbers of the statement** are per member, so a group of `k = 4` needs:

| Item | Required |
|---|---|
| Manual publications | at least 5 per member (20), at least 20 matched pairs each |
| Manual matched pairs | at least 100 per member (400) |
| Automatic publications | at least 10% of **each member's** publications outside the manual pool, at least `2k` = 8, never a hand-labelled publication |
| Test, and valid | `k` = 4 manual **and** `k` = 4 automatic publications each; train is the rest |
| `predictions` | one row per entry **with a true reference** of every train / valid / test publication |
| MRR@5 | over the entries with a true reference only (`nil_mode_main` = `skip` in `logs/m3_metrics.json`) |

A label is a matched pair (entry, reference) (teacher's QA, M3 question 4, 27 Nov): an entry whose reference is not
among the arXiv references of its publication makes no pair, has no row in `predictions` and is not in the MRR, so
a publication without any matched entry has no row in `predictions`. The manual publications that were replaced
(`labels/manual_old/`) are hand-labelled all the same: they must have no automatic label.

## 0. Setup

**On Colab** (CPU runtime): open this notebook and *Run all*. It mounts Drive, runs the offline tests and then
checks the real files without writing anything. `25_run_m3` can load it with `RUN_TEST = False` and call
`run_analysis()`, which also writes `logs/m3_requirements.json`.

In [ ]:
import contextlib
import io
import json
import re
import tempfile
import unittest
from pathlib import Path

import pandas as pd

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if "RUN_TEST" not in globals():
    RUN_TEST = True

if "PROJECT_ROOT" not in globals():
    if ON_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive/arxiv-project")
    else:   # notebooks live in src/, so the repo root is one level up
        PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

# From 20 when it is loaded; else a copy of 20's table (the course's registration sheet, group 10).
MEMBER_RANGES = globals().get("MEMBER_RANGES", {
    "TV1": ("2603.20580", "2603.25579"),
    "TV2": ("2609.32834", "2609.37833"),
    "TV3": ("2605.17338", "2605.22337"),
    "TV4": ("2605.07338", "2605.12337"),
})
GROUP_SIZE = len(MEMBER_RANGES)
MANUAL_PUBS_PER_MEMBER = 5
PAIRS_PER_MANUAL_PUB = 20          # matched (entry, reference) pairs, not entries (teacher's QA)
MANUAL_PAIRS_PER_MEMBER = 100
AUTO_MIN_SHARE = 0.10              # of each member's publications outside the manual pool
AUTO_PUBS_PER_MEMBER = 2
SPLIT_PUBS_PER_MEMBER = 1          # manual and automatic publications per member in test, and in valid

KEY = ["arxiv_id", "citation_key"]
PARTITIONS = ("train", "valid", "test")
CANDIDATES = [f"cand_{i}" for i in range(1, 6)]
PREDICTION_COLUMNS = ["arxiv_id", "partition", "citation_key", "groundtruth_arxiv_id"] + CANDIDATES
NIL_MODE_MAIN = "skip"             # MRR@5 over the entries with a true reference only (23's NIL_MODE)
print(f"24 loaded | project: {PROJECT_ROOT} | group of {GROUP_SIZE}")

## 1. Inputs

`ARTIFACTS` is the data contract as this notebook reads it. Only the columns the checks need are read from the
large tables. A file that is absent or broken never stops the notebook.

`labels/manual/*.csv` are read as text, and a `ref_arxiv_id` cell is read as `21` reads it (`label_id`): empty
means no match, `h1`/`h2`/`h3` is that hint, a pasted hint or an arXiv URL gives its ID, and a number that lost
zeros in a spreadsheet (`2504.1`) is padded again. `validate_labels` of `21` stays the official check.
Of `labels/manual_old/*.csv` (replaced manual publications) only the file names are read.

In [ ]:
ARTIFACTS = {   # name: (path under the project folder, owner, written by, columns to read or None for all)
    "entries": ("data/m3/entries.parquet", "TV1", "20_m3_data", KEY),
    "refs": ("data/m3/refs.parquet", "TV1", "20_m3_data", ["arxiv_id", "ref_arxiv_id"]),
    "pubs": ("data/m3/pubs.parquet", "TV1", "20_m3_data", None),
    "m3_data_summary": ("logs/m3_data_summary.json", "TV1", "20_m3_data", None),
    "m2_stats": ("logs/m2_stats.json", "TV1", "15_run_parsing", None),
    "m2_benchmark": ("logs/m2_benchmark.json", "TV1", "15_run_parsing", None),
    "selection": ("labels/selection.csv", "TV2", "21_labeling", None),
    "auto_labels": ("labels/auto/auto_labels.parquet", "TV2", "21_labeling", None),
    "split": ("labels/split.json", "TV2", "21_labeling", None),
    "labels": ("labels/labels.parquet", "TV2", "21_labeling", None),
    "label_report": ("logs/m3_labels.json", "TV2", "21_labeling", None),
    "pairs": ("data/m3/pairs.parquet", "TV3", "22_features", KEY + ["label", "source", "partition"]),
    "feature_report": ("logs/m3_feature_report.csv", "TV3", "22_features", None),
    "cleaning_report": ("logs/m3_cleaning_report.csv", "TV3", "22_features", None),
    "predictions": ("data/m3/predictions.parquet", "TV3", "23_model", None),
    "metrics": ("logs/m3_metrics.json", "TV3", "23_model", None),
    "model_selection": ("logs/m3_model_selection.csv", "TV3", "23_model", None),
}
_FID_RE = re.compile(r"(\d{4})-(\d{4,5})")
_HINT_RE = re.compile(r"^h?([1-9])$", re.I)
_ID_PREFIX_RE = re.compile(r"^(?:(?:https?://)?(?:www\.|export\.)?arxiv\.org/(?:abs|pdf)/|arxiv:\s*)", re.I)
_NUMERIC_ID_RE = re.compile(r"^(\d{3,4})\.(\d{1,5})$")
_NO_MATCH = {"", "nan", "none", "no", "-", "0"}


def _read_file(path, columns=None):
    if path.suffix == ".parquet":
        return pd.read_parquet(path, columns=columns)
    if path.suffix == ".json":
        return json.loads(path.read_text(encoding="utf-8"))
    return pd.read_csv(path, dtype={"arxiv_id": str}, encoding="utf-8-sig")


def label_id(value, hints=()):
    """A `ref_arxiv_id` cell -> arXiv ID, None (no match), or the text when it is neither (as `21` reads it)."""
    text = str(value).strip().strip("'\"")
    if text.lower() in _NO_MATCH:
        return None
    hint = _HINT_RE.match(text)
    if hint:
        k = int(hint.group(1)) - 1
        return label_id(hints[k].split("|")[0]) if k < len(hints) and hints[k].strip() else text
    text = _ID_PREFIX_RE.sub("", text.split("|")[0].strip())
    text = re.sub(r"(?<=\d)(?:v\d+)?(?:\.pdf)?$", "", text, flags=re.I)
    number = _NUMERIC_ID_RE.match(text)
    if number:   # lost leading zero (704.0001) or trailing zeros (2504.1)
        month = number.group(1).zfill(4)
        text = f"{month}.{number.group(2).ljust(5 if int(month) >= 1501 else 4, '0')}"
    return text


def read_manual_sheets(root=None):
    """(labels, problems) of labels/manual/*.csv: arxiv_id, citation_key, ref_arxiv_id (None = no match), file.

    The publication comes from the file name (`2603-20581.csv`). `<fid>_refs.csv` files are skipped.
    """
    folder = Path(root or PROJECT_ROOT) / "labels" / "manual"
    frames, problems = [], []
    for path in sorted(folder.glob("*.csv")):
        if path.name.endswith("_refs.csv"):
            continue
        match = _FID_RE.search(path.name)
        try:
            df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        except (OSError, ValueError) as err:
            problems.append(f"{path.name}: {type(err).__name__}: {err}")
            continue
        df.columns = [c.strip() for c in df.columns]
        if not match or not {"citation_key", "ref_arxiv_id"} <= set(df.columns):
            problems.append(f"{path.name}: needs a publication id in the name and citation_key, ref_arxiv_id columns")
            continue
        df = df[df["citation_key"].str.strip() != ""]
        hint_columns = [c for c in ("hint_1", "hint_2", "hint_3") if c in df.columns]
        hints = df[hint_columns].to_numpy().tolist()
        refs = [label_id(value, hint) for value, hint in zip(df["ref_arxiv_id"], hints)]
        frames.append(pd.DataFrame({"arxiv_id": f"{match.group(1)}.{match.group(2)}",
                                    "citation_key": df["citation_key"].str.strip().to_numpy(),
                                    "ref_arxiv_id": pd.Series(refs, dtype=object), "file": path.name}))
    columns = ["arxiv_id", "citation_key", "ref_arxiv_id", "file"]
    labels = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
    return labels[columns], problems


def read_manual_old(root=None):
    """The publications of labels/manual_old/*.csv (replaced manual ones), or None without that folder."""
    folder = Path(root or PROJECT_ROOT) / "labels" / "manual_old"
    if not folder.is_dir():
        return None
    return sorted({f"{m.group(1)}.{m.group(2)}" for m in (_FID_RE.search(p.name) for p in folder.glob("*.csv")) if m})


def load_m3_artifacts(root=None, names=None):
    """(data, status): data[name] is the table or document, or None; status has one row per file."""
    root = Path(root or PROJECT_ROOT)
    data, status = {}, []
    for name in names or ARTIFACTS:
        relative, owner, writer, columns = ARTIFACTS[name]
        path = root / relative
        value, state, error = None, "missing", ""
        if path.is_file():
            try:
                value, state = _read_file(path, columns), "ok"
            except Exception as err:   # a broken or half-written file: report it, go on
                state, error = "unreadable", f"{type(err).__name__}: {err}"[:200]
        data[name] = value
        rows = len(value) if isinstance(value, pd.DataFrame) else None
        status.append({"name": name, "state": state, "rows": rows, "owner": owner, "written_by": writer,
                       "path": relative, "error": error})
    if names is None:
        manual, problems = read_manual_sheets(root)
        data["manual_sheets"] = manual if len(manual) else None
        status.append({"name": "manual_sheets", "state": "unreadable" if problems else "ok" if len(manual) else "missing",
                       "rows": len(manual), "owner": "all", "written_by": "each member", "path": "labels/manual/*.csv",
                       "error": "; ".join(problems)[:500]})
        old = read_manual_old(root)
        data["manual_old"] = old
        status.append({"name": "manual_old", "state": "missing" if old is None else "ok",
                       "rows": None if old is None else len(old), "owner": "all", "written_by": "each member",
                       "path": "labels/manual_old/*.csv", "error": ""})
    status = pd.DataFrame(status)
    status["rows"] = status["rows"].astype("Int64")
    return data, status


def print_inputs(status):
    counts = status["state"].value_counts().to_dict()
    print(f"[24] inputs: {counts}")
    for row in status.itertuples(index=False):
        rows = "" if pd.isna(row.rows) else f"{row.rows:,} rows"
        print(f"  {row.state:<10} {row.path:<34} {rows:>14}  ({row.owner}, {row.written_by}) {row.error}")

## 2. Statement requirements

`check_requirements(data)` gives one row per rule with `status`:

- `PASS` / `FAIL`: the rule was checked;
- `WAIT`: an input file is still missing.

Labels come from `labels/labels.parquet` (after `run_labeling`); while it does not exist, from the manual sheets
and `auto_labels.parquet`, so the manual quotas can be followed during labelling. The manual quotas count
matched pairs (entries with a `ref_arxiv_id`), as `validate_labels` of `21` does.

In [ ]:
def member_of(arxiv_id):
    """The member whose ID range holds the ID, or None."""
    def number(text):
        month, _, digits = str(text).partition(".")
        return (month, int(digits)) if digits.isdigit() else None
    key = number(arxiv_id)
    if key is None:
        return None
    for member, (first, last) in MEMBER_RANGES.items():
        if number(first) <= key <= number(last):
            return member
    return None


def _clean_ids(frame, columns):
    """A copy with `columns` as plain objects, None where missing."""
    out = frame.copy()
    for column in columns:
        values = out[column].astype(object)
        out[column] = values.where(values.notna() & (values != ""), None)
    return out


def labels_for_check(data):
    """(labels with a `source` column, where they come from) or (None, "")."""
    if data.get("labels") is not None:
        labels = data["labels"][KEY + ["ref_arxiv_id", "source"]]
        return _clean_ids(labels.astype({"arxiv_id": str, "source": str}), ["ref_arxiv_id"]), "labels.parquet"
    frames, origin = [], []
    for name, source in (("manual_sheets", "manual"), ("auto_labels", "auto")):
        if data.get(name) is not None:
            frames.append(data[name][KEY + ["ref_arxiv_id"]].assign(source=source))
            origin.append(name)
    if not frames:
        return None, ""
    labels = _clean_ids(pd.concat(frames, ignore_index=True).astype({"arxiv_id": str}), ["ref_arxiv_id"])
    manual = set(labels.loc[labels["source"] == "manual", "arxiv_id"])   # manual labels win, as in 21
    return labels[(labels["source"] == "manual") | ~labels["arxiv_id"].isin(manual)], " + ".join(origin)


def _split_ids(split):
    """split.json -> {(partition, source): [arxiv_id, ...]}; a value may be one ID or a list."""
    out = {}
    for partition in ("test", "valid"):
        for source in ("manual", "auto"):
            ids = (split.get(partition) or {}).get(source) or []
            out[partition, source] = [str(a) for a in ([ids] if isinstance(ids, str) else ids)]
    return out


def _examples(values, n=5):
    values = sorted(values)
    return "" if not values else f"{len(values)}: {values[:n]}"


def check_requirements(data):
    """One row per rule: check, expected, actual, status (PASS / FAIL / WAIT), detail."""
    rows = []

    def add(check, expected, ok, actual, detail=""):
        rows.append({"check": check, "expected": expected, "actual": str(actual),
                     "status": "PASS" if ok else "FAIL", "detail": "" if ok else detail})

    def wait(check, expected, needs):
        rows.append({"check": check, "expected": expected, "actual": "-", "status": "WAIT",
                     "detail": f"needs {needs}"})

    k = GROUP_SIZE
    labels, origin = labels_for_check(data)
    entries, refs, pubs = data.get("entries"), data.get("refs"), data.get("pubs")
    entry_keys = ref_keys = None
    if entries is not None:
        entry_keys = set(zip(entries["arxiv_id"].astype(str), entries["citation_key"]))
    if refs is not None:
        ref_keys = set(zip(refs["arxiv_id"].astype(str), refs["ref_arxiv_id"].astype(str)))
    manual_ids, auto_ids = set(), set()
    if labels is not None:
        manual_ids = set(labels.loc[labels["source"] == "manual", "arxiv_id"])
        auto_ids = set(labels.loc[labels["source"] == "auto", "arxiv_id"])
    labelled_keys = set() if labels is None else set(zip(labels["arxiv_id"], labels["citation_key"]))

    # ---------- manual labels: matched pairs (entry, reference), as the teacher's QA counts them ----------
    manual_rules = [("manual_pubs", f">= {MANUAL_PUBS_PER_MEMBER * k}"),
                    ("manual_pubs_per_member", f">= {MANUAL_PUBS_PER_MEMBER} each"),
                    ("manual_pairs_per_pub", f">= {PAIRS_PER_MANUAL_PUB} matched pairs"),
                    ("manual_pairs_total", f">= {MANUAL_PAIRS_PER_MEMBER * k} matched pairs")]
    if not manual_ids:
        for check, expected in manual_rules:
            wait(check, expected, "manual labels (labels/manual/*.csv)")
    else:
        manual = labels[labels["source"] == "manual"]
        pairs_per_pub = (manual[manual["ref_arxiv_id"].notna()].groupby("arxiv_id").size()
                         .reindex(sorted(manual_ids), fill_value=0))
        per_member = {m: 0 for m in MEMBER_RANGES}
        for arxiv_id in manual_ids:
            member = member_of(arxiv_id)
            if member:
                per_member[member] += 1
        short = {a: int(n) for a, n in pairs_per_pub.items() if n < PAIRS_PER_MANUAL_PUB}
        add(*manual_rules[0], len(manual_ids) >= MANUAL_PUBS_PER_MEMBER * k, len(manual_ids), f"read from {origin}")
        add(*manual_rules[1], min(per_member.values()) >= MANUAL_PUBS_PER_MEMBER, per_member, f"read from {origin}")
        add(*manual_rules[2], not short, int(pairs_per_pub.min()), f"too few: {short}")
        add(*manual_rules[3], pairs_per_pub.sum() >= MANUAL_PAIRS_PER_MEMBER * k,
            f"{int(pairs_per_pub.sum())} pairs in {len(manual)} entries", f"read from {origin}")

    # ---------- every entry once, references of the same publication ----------
    if labels is None:
        wait("one_label_per_entry", "no duplicate", "labels")
        wait("every_entry_labelled", "all entries of each labelled publication", "labels")
        wait("labels_point_to_own_refs", "references of the same publication", "labels")
    else:
        twice = labels[labels.duplicated(KEY, keep=False)]
        add("one_label_per_entry", "no duplicate", twice.empty, len(twice),
            _examples(set(zip(twice["arxiv_id"], twice["citation_key"]))))
        if entry_keys is None:
            wait("every_entry_labelled", "all entries of each labelled publication", "data/m3/entries.parquet")
        else:
            expected_keys = {key for key in entry_keys if key[0] in manual_ids | auto_ids}
            missing, unknown = expected_keys - labelled_keys, labelled_keys - entry_keys
            add("every_entry_labelled", "all entries of each labelled publication", not missing and not unknown,
                f"{len(missing)} missing, {len(unknown)} unknown",
                f"missing {_examples(missing)} unknown {_examples(unknown)}")
        if ref_keys is None:
            wait("labels_point_to_own_refs", "references of the same publication", "data/m3/refs.parquet")
        else:
            positive = labels[labels["ref_arxiv_id"].notna()]
            foreign = set(zip(positive["arxiv_id"], positive["ref_arxiv_id"])) - ref_keys
            add("labels_point_to_own_refs", "references of the same publication", not foreign, len(foreign),
                _examples(foreign))

    # ---------- automatic labels ----------
    hand = manual_ids | set(data.get("manual_old") or [])       # hand-labelled, now or before a replacement
    selection = data.get("selection")
    if isinstance(selection, pd.DataFrame) and "arxiv_id" in selection.columns:
        hand |= set(selection["arxiv_id"].astype(str))
    share_each = f">= {AUTO_MIN_SHARE:.0%} each"
    if not auto_ids:
        wait("auto_pubs_min", f">= {AUTO_PUBS_PER_MEMBER * k}", "automatic labels")
        wait("auto_not_hand_labelled", "no hand-labelled publication", "automatic labels")
        wait("auto_share", f">= {AUTO_MIN_SHARE:.0%}", "automatic labels")
        wait("auto_share_per_member", share_each, "automatic labels")
    else:
        add("auto_pubs_min", f">= {AUTO_PUBS_PER_MEMBER * k}", len(auto_ids) >= AUTO_PUBS_PER_MEMBER * k, len(auto_ids))
        both = auto_ids & hand
        add("auto_not_hand_labelled", "no hand-labelled publication", not both, len(both),
            f"{_examples(both)} (in labels/manual, labels/manual_old or selection.csv)")
        if pubs is None:
            wait("auto_share", f">= {AUTO_MIN_SHARE:.0%}", "data/m3/pubs.parquet")
            wait("auto_share_per_member", share_each, "data/m3/pubs.parquet")
        else:   # the statement's denominator: every publication outside the manual pool, eligible or not
            others = set(pubs["arxiv_id"].astype(str)) - manual_ids
            share = len(auto_ids & others) / len(others) if others else 0.0
            add("auto_share", f">= {AUTO_MIN_SHARE:.0%}", share >= AUTO_MIN_SHARE, f"{share:.1%}",
                f"{len(auto_ids & others):,} of {len(others):,} publications outside the manual pool")
            mine = {m: set() for m in MEMBER_RANGES}
            for arxiv_id in others:
                member = member_of(arxiv_id)
                if member:
                    mine[member].add(arxiv_id)
            per_member = {m: len(auto_ids & ids) / len(ids) if ids else 0.0 for m, ids in mine.items()}
            low = {m: f"{v:.1%}" for m, v in per_member.items() if v < AUTO_MIN_SHARE}
            add("auto_share_per_member", share_each, not low, {m: f"{v:.1%}" for m, v in per_member.items()},
                f"below {AUTO_MIN_SHARE:.0%}: {low}")

    # ---------- split ----------
    split_checks = [(f"split_{p}_{s}", f"= {SPLIT_PUBS_PER_MEMBER * k}") for p in ("test", "valid")
                    for s in ("manual", "auto")]
    partition_of = {}
    if data.get("split") is None:
        for check, expected in split_checks + [("split_consistent", "disjoint, labelled, source matches")]:
            wait(check, expected, "labels/split.json")
    else:
        parts = _split_ids(data["split"])
        for (check, expected), key in zip(split_checks, parts):
            add(check, expected, len(set(parts[key])) == SPLIT_PUBS_PER_MEMBER * k, len(set(parts[key])),
                f"{parts[key]} (the statement counts {SPLIT_PUBS_PER_MEMBER} per member)")
        listed = [a for ids in parts.values() for a in ids]
        partition_of = {a: partition for (partition, _), ids in parts.items() for a in ids}
        if labels is None:
            wait("split_consistent", "disjoint, labelled, source matches", "labels")
        else:
            repeated = {a for a in listed if listed.count(a) > 1}
            wrong = {a for (_, source), ids in parts.items() for a in ids
                     if a not in (manual_ids if source == "manual" else auto_ids)}
            train_manual, train_auto = manual_ids - set(listed), auto_ids - set(listed)
            add("split_consistent", "disjoint, labelled, source matches",
                not repeated and not wrong and bool(train_manual) and bool(train_auto),
                f"train: {len(train_manual)} manual, {len(train_auto)} auto",
                f"repeated {sorted(repeated)}; not labelled by that source {sorted(wrong)}")

    # ---------- MRR: the entries with a true reference only ----------
    metrics = data.get("metrics")
    if not isinstance(metrics, dict):
        wait("mrr_nil_mode", f"= {NIL_MODE_MAIN}", "logs/m3_metrics.json")
    else:
        mode = metrics.get("nil_mode_main")
        test = ((metrics.get("final") or {}).get("test") or {}).get("all") or {}
        value = test.get(f"mrr5_{mode}")
        add("mrr_nil_mode", f"= {NIL_MODE_MAIN}", mode == NIL_MODE_MAIN,
            mode if not isinstance(value, (int, float)) else f"{mode} (test MRR@5 {value:.4f})",
            "MRR@5 must count only the entries with a true reference (NIL_MODE of 23)")

    # ---------- predictions: one row per entry with a true reference ----------
    names = ["pred_columns", "pred_publications", "pred_rows", "pred_partition", "pred_groundtruth", "pred_candidates"]
    expectations = ["the columns of the statement", "labelled publications with a pair",
                    "one row per entry with a true reference", "as split.json", "as the labels",
                    "own references, no repeat, no gap"]
    predictions = data.get("predictions")
    if predictions is None:
        for check, expected in zip(names, expectations):
            wait(check, expected, "data/m3/predictions.parquet")
        return pd.DataFrame(rows)
    good_columns = list(predictions.columns) == PREDICTION_COLUMNS
    add(names[0], expectations[0], good_columns, len(predictions.columns), f"got {list(predictions.columns)}")
    if not good_columns:
        for check, expected in zip(names[1:], expectations[1:]):
            wait(check, expected, "a predictions table with the right columns")
        return pd.DataFrame(rows)
    p = _clean_ids(predictions.astype({"arxiv_id": object}), ["groundtruth_arxiv_id", "partition"] + CANDIDATES)
    p["arxiv_id"] = p["arxiv_id"].astype(str)
    published = set(p["arxiv_id"])
    row_keys = set(zip(p["arxiv_id"], p["citation_key"]))
    if labels is None:
        wait(names[1], expectations[1], "labels")
        wait(names[2], expectations[2], "labels")
    else:
        labelled = manual_ids | auto_ids
        matched = labels[labels["ref_arxiv_id"].notna()]
        with_pair = set(matched["arxiv_id"]) & labelled
        missing, extra = with_pair - published, published - labelled
        add(names[1], expectations[1], not missing and not extra,
            f"{len(published)} ({len(labelled - with_pair)} labelled without a pair)",
            f"missing {_examples(missing)} not labelled {_examples(extra)}")
        gold_keys = set(zip(matched["arxiv_id"], matched["citation_key"]))
        missing, no_truth, doubled = gold_keys - row_keys, row_keys - gold_keys, int(p.duplicated(KEY).sum())
        unknown = len(row_keys - entry_keys) if entry_keys is not None else "?"
        add(names[2], expectations[2], not missing and not no_truth and not doubled, f"{len(p):,} rows",
            f"missing {_examples(missing)} without a true reference {_examples(no_truth)} (not an entry: {unknown})"
            f" duplicated {doubled}")
    if data.get("split") is None:
        wait(names[3], expectations[3], "labels/split.json")
    else:
        wrong = {a for a, part in zip(p["arxiv_id"], p["partition"]) if part != partition_of.get(a, "train")}
        add(names[3], expectations[3], not wrong, p.drop_duplicates("arxiv_id")["partition"].value_counts().to_dict(),
            f"wrong partition {_examples(wrong)}")
    if labels is None:
        wait(names[4], expectations[4], "labels")
    else:
        gold = {(a, c): r for a, c, r in labels[KEY + ["ref_arxiv_id"]].itertuples(index=False)}
        wrong = {(a, c) for a, c, g in zip(p["arxiv_id"], p["citation_key"], p["groundtruth_arxiv_id"])
                 if (a, c) in gold and g != gold[a, c]}
        add(names[4], expectations[4], not wrong, f"{len(wrong)} differ", _examples(wrong))
    bad = set()
    for record in p[KEY + CANDIDATES].itertuples(index=False):
        arxiv_id, key, *cands = record
        given = [c for c in cands if c is not None]
        gap = cands[:len(given)] != given
        foreign = ref_keys is not None and any((arxiv_id, c) not in ref_keys for c in given)
        if gap or foreign or len(set(given)) != len(given):
            bad.add((arxiv_id, key))
    add(names[5], expectations[5], not bad, f"{len(bad)} bad rows",
        _examples(bad) + ("" if ref_keys is not None else " (references not checked: refs.parquet missing)"))
    return pd.DataFrame(rows)


def print_checks(checks):
    print(f"[24] requirements: {checks['status'].value_counts().to_dict()}")
    for row in checks.itertuples(index=False):
        detail = f" | {row.detail}" if row.detail and row.status != "PASS" else ""
        print(f"  {row.status:<4} {row.check:<26} expected {row.expected:<40} got {row.actual}{detail[:300]}")

## 3. Run

`run_analysis()` loads the inputs, checks the requirements, prints both and (with `save=True`) writes
`logs/m3_requirements.json`.

In [ ]:
def run_analysis(root=None, save=True):
    """{"data", "inputs", "checks", "ok"}; ok means every rule is PASS."""
    root = Path(root or PROJECT_ROOT)
    data, status = load_m3_artifacts(root)
    checks = check_requirements(data)
    print_inputs(status)
    print_checks(checks)
    ok = bool((checks["status"] == "PASS").all())
    if save:
        path = root / "logs" / "m3_requirements.json"
        path.parent.mkdir(parents=True, exist_ok=True)
        report = {"ok": ok, "inputs": json.loads(status.to_json(orient="records")),
                  "checks": checks.to_dict("records")}
        path.write_text(json.dumps(report, indent=1), encoding="utf-8")
        print(f"[24] wrote {path}")
    return {"data": data, "inputs": status, "checks": checks, "ok": ok}

## 4. Tests

Offline, on a toy group that satisfies every rule (`_toy`): 20 manual and 40 automatic publications of 24
entries (20 matched, 4 without a reference) and 20 references, a 4 + 4 / 4 + 4 split, predictions for the matched
entries and `nil_mode_main` = `skip`. Each test breaks one thing. The
Parquet test is skipped where no Parquet engine is installed (Colab has `pyarrow`).

In [ ]:
def _toy():
    entries, refs, pubs, labels, manual_ids, auto_ids = [], [], [], [], {}, {}
    for member, (first, _) in MEMBER_RANGES.items():
        month, number = first.split(".")
        ids = [f"{month}.{int(number) + i:05d}" for i in range(30)]   # 5 manual, 10 automatic, 15 unlabelled
        manual_ids[member], auto_ids[member] = ids[:5], ids[5:15]
        for i, arxiv_id in enumerate(ids):
            pubs.append({"arxiv_id": arxiv_id, "member": member, "eligible": True})
            ref_ids = [f"2301.{10000 + j}" for j in range(20)]
            refs += [{"arxiv_id": arxiv_id, "ref_arxiv_id": r} for r in ref_ids]
            for j in range(24):                     # 20 matched pairs, 4 entries without a reference
                entries.append({"arxiv_id": arxiv_id, "citation_key": f"k{j}"})
                if i < 15:
                    labels.append({"arxiv_id": arxiv_id, "citation_key": f"k{j}",
                                   "ref_arxiv_id": ref_ids[j] if j < 20 else None,
                                   "source": "manual" if i < 5 else "auto"})
    split = {"seed": 42,
             "test": {"manual": [v[0] for v in manual_ids.values()], "auto": [v[0] for v in auto_ids.values()]},
             "valid": {"manual": [v[1] for v in manual_ids.values()], "auto": [v[1] for v in auto_ids.values()]}}
    labels = pd.DataFrame(labels)
    partition_of = {a: p for p in ("test", "valid") for ids in split[p].values() for a in ids}
    predictions = pd.DataFrame(
        [[a, partition_of.get(a, "train"), c, g, "2301.10000", "2301.10001", None, None, None]
         for a, c, g in labels[KEY + ["ref_arxiv_id"]].itertuples(index=False) if isinstance(g, str)],
        columns=PREDICTION_COLUMNS)                 # only the entries with a true reference
    metrics = {"nil_mode_main": "skip", "final": {"test": {"all": {"mrr5_skip": 0.9, "mrr5_zero": 0.7}}}}
    return {"entries": pd.DataFrame(entries), "refs": pd.DataFrame(refs), "pubs": pd.DataFrame(pubs),
            "labels": labels, "split": split, "predictions": predictions, "metrics": metrics}


class AnalysisTests(unittest.TestCase):
    def status(self, data):
        return dict(check_requirements(data)[["check", "status"]].itertuples(index=False))

    def failed(self, data):
        return {check for check, status in self.status(data).items() if status == "FAIL"}

    def test_complete_group_passes(self):
        checks = check_requirements(_toy())
        self.assertEqual(set(checks["status"]), {"PASS"}, checks[checks["status"] != "PASS"].to_string())

    def test_member_ranges(self):
        self.assertEqual(member_of("2605.07338"), "TV4")
        self.assertEqual(member_of("2605.12337"), "TV4")
        self.assertEqual(member_of("2603.20580"), "TV1")
        self.assertIsNone(member_of("2605.12338"))
        self.assertIsNone(member_of("hep-th/9901001"))

    def test_one_plus_one_split_fails(self):
        data = _toy()
        data["split"] = {"seed": 42, **{p: {s: data["split"][p][s][0] for s in ("manual", "auto")}
                                        for p in ("test", "valid")}}
        self.assertLessEqual({"split_test_manual", "split_test_auto", "split_valid_manual", "split_valid_auto"},
                             self.failed(data))

    def test_publication_in_two_partitions(self):
        data = _toy()
        data["split"]["valid"]["manual"] = list(data["split"]["test"]["manual"])
        self.assertIn("split_consistent", self.failed(data))

    def test_manual_quotas(self):
        data = _toy()
        labels = data["labels"]
        dropped = labels[labels["source"] == "manual"]["arxiv_id"].iloc[-1]    # one manual publication
        data["labels"] = labels[labels["arxiv_id"] != dropped]
        self.assertLessEqual({"manual_pubs", "manual_pubs_per_member", "manual_pairs_total"}, self.failed(data))

    def test_pairs_are_counted_not_entries(self):
        data = _toy()   # 24 entries but 19 matched pairs in one manual publication
        labels, p = data["labels"], data["predictions"]
        first = labels[labels["source"] == "manual"]["arxiv_id"].iloc[0]
        labels.loc[(labels["arxiv_id"] == first) & (labels["citation_key"] == "k0"), "ref_arxiv_id"] = None
        data["predictions"] = p[~((p["arxiv_id"] == first) & (p["citation_key"] == "k0"))]
        self.assertEqual(self.failed(data), {"manual_pairs_per_pub", "manual_pairs_total"})

    def test_predictions_only_for_entries_with_a_reference(self):
        data = _toy()   # a row for an entry without a reference, as before the QA
        p = data["predictions"]
        extra = p.iloc[[0]].assign(citation_key="k20", groundtruth_arxiv_id=None)
        data["predictions"] = pd.concat([p, extra], ignore_index=True)
        self.assertEqual(self.failed(data), {"pred_rows"})

    def test_publication_without_a_pair_has_no_row(self):
        data = _toy()
        labels, p = data["labels"], data["predictions"]
        no_pair = sorted(labels[labels["source"] == "auto"]["arxiv_id"].unique())[-1]   # not in the split
        labels.loc[labels["arxiv_id"] == no_pair, "ref_arxiv_id"] = None
        data["predictions"] = p[p["arxiv_id"] != no_pair]
        self.assertEqual(self.failed(data), set())

    def test_auto_share_per_member(self):
        data = _toy()   # the first member keeps only its 2 automatic publications of the split: 2 / 25 = 8%
        labels, p, first = data["labels"], data["predictions"], next(iter(MEMBER_RANGES))
        in_split = set(data["split"]["test"]["auto"]) | set(data["split"]["valid"]["auto"])
        drop = {a for a in labels.loc[labels["source"] == "auto", "arxiv_id"] if member_of(a) == first} - in_split
        data["labels"], data["predictions"] = labels[~labels["arxiv_id"].isin(drop)], p[~p["arxiv_id"].isin(drop)]
        self.assertEqual(self.failed(data), {"auto_share_per_member"})

    def test_hand_labelled_publication_has_no_automatic_label(self):
        for name in ("manual_old", "selection"):
            data = _toy()
            auto_id = data["split"]["test"]["auto"][0]
            data[name] = [auto_id] if name == "manual_old" else pd.DataFrame({"member": ["TV1"], "arxiv_id": [auto_id]})
            self.assertEqual(self.failed(data), {"auto_not_hand_labelled"}, name)

    def test_mrr_over_entries_with_a_reference(self):
        data = _toy()
        data["metrics"] = {**data["metrics"], "nil_mode_main": "zero"}
        self.assertEqual(self.failed(data), {"mrr_nil_mode"})

    def test_unlabelled_entry(self):
        data = _toy()
        data["labels"] = data["labels"].iloc[1:]
        failed = self.failed(data)
        self.assertLessEqual({"every_entry_labelled", "manual_pairs_per_pub"}, failed)

    def test_label_outside_the_publication(self):
        data = _toy()
        data["labels"].loc[0, "ref_arxiv_id"] = "9999.99999"
        self.assertIn("labels_point_to_own_refs", self.failed(data))

    def test_too_few_automatic_publications(self):
        data = _toy()
        labels = data["labels"]
        keep = sorted(labels[labels["source"] == "auto"]["arxiv_id"].unique())[:3]
        data["labels"] = labels[(labels["source"] == "manual") | labels["arxiv_id"].isin(keep)]
        self.assertIn("auto_pubs_min", self.failed(data))

    def test_auto_share_counts_every_publication(self):
        data = _toy()
        more = pd.DataFrame([{"arxiv_id": f"2605.{30000 + i}", "member": None, "eligible": False} for i in range(900)])
        data["pubs"] = pd.concat([data["pubs"], more], ignore_index=True)
        self.assertIn("auto_share", self.failed(data))

    def test_label_cells(self):
        hints = ["2301.10140 | A title", "", "hep-th/9901001 | Old"]
        for cell, expected in (("", None), ("-", None), ("h1", "2301.10140"), ("1", "2301.10140"), ("H3", "hep-th/9901001"),
                               ("h2", "h2"), ("2301.10140 | A title", "2301.10140"), ("2301.1014", "2301.10140"),
                               ("704.0001", "0704.0001"), ("arXiv:2301.10140v2", "2301.10140"),
                               ("https://arxiv.org/pdf/2301.10140v2.pdf", "2301.10140"), ("not an id", "not an id")):
            self.assertEqual(label_id(cell, hints), expected, cell)

    def test_prediction_errors(self):
        for check, change in (
                ("pred_rows", lambda p: p.iloc[1:]),
                ("pred_partition", lambda p: p.assign(partition="train")),
                ("pred_groundtruth", lambda p: p.assign(groundtruth_arxiv_id=None)),
                ("pred_candidates", lambda p: p.assign(cand_2=p["cand_1"])),
                ("pred_candidates", lambda p: p.assign(cand_4="2301.10003")),
                ("pred_columns", lambda p: p.drop(columns="cand_5"))):
            data = _toy()
            data["predictions"] = change(data["predictions"])
            self.assertIn(check, self.failed(data), check)

    def test_string_dtype_predictions_pass(self):
        data = _toy()
        data["predictions"] = data["predictions"].astype("string")   # the types 23 writes
        self.assertEqual(self.failed(data), set())

    def test_nothing_there_yet(self):
        checks = check_requirements({})
        self.assertEqual(set(checks["status"]), {"WAIT"})
        with tempfile.TemporaryDirectory() as tmp:
            data, status = load_m3_artifacts(tmp)
            self.assertEqual(set(status["state"]), {"missing"})
            self.assertEqual(set(check_requirements(data)["status"]), {"WAIT"})

    def test_loader_and_manual_sheets(self):
        with tempfile.TemporaryDirectory() as tmp:
            manual = Path(tmp) / "labels" / "manual"
            manual.mkdir(parents=True)
            (manual / "2605-08524 - Sheet1.csv").write_text(
                "citation_key,hint_1,ref_arxiv_id,confidence\na,2301.10140 | T,h1,sure\nb,2301.10140 | T,,sure\n,,,\n",
                encoding="utf-8-sig")
            (manual / "2605-08524_refs.csv").write_text("ref_arxiv_id,ref_title\n2301.10140,T\n", encoding="utf-8")
            (manual.parent / "selection.csv").write_text("member,arxiv_id\nTV4,2605.08800\n", encoding="utf-8")
            (manual.parent / "split.json").write_text("{not json", encoding="utf-8")
            (manual.parent / "manual_old").mkdir()
            (manual.parent / "manual_old" / "2605-09000 - 2605-09000.csv").write_text("citation_key\n", encoding="utf-8")
            data, status = load_m3_artifacts(tmp)
            state = dict(zip(status["name"], status["state"]))
            self.assertEqual((state["selection"], state["split"], state["manual_sheets"]), ("ok", "unreadable", "ok"))
            self.assertEqual((state["manual_old"], data["manual_old"]), ("ok", ["2605.09000"]))
            self.assertEqual(data["selection"]["arxiv_id"].tolist(), ["2605.08800"])      # the zeros are kept
            labels, origin = labels_for_check(data)
            self.assertEqual(origin, "manual_sheets")
            self.assertEqual(labels[KEY].values.tolist(), [["2605.08524", "a"], ["2605.08524", "b"]])
            self.assertEqual(labels["ref_arxiv_id"].tolist(), ["2301.10140", None])
            self.assertEqual(self.status(data)["manual_pairs_per_pub"], "FAIL")

    def test_run_writes_the_report(self):
        with tempfile.TemporaryDirectory() as tmp:
            with contextlib.redirect_stdout(io.StringIO()):
                result = run_analysis(tmp, save=True)
            report = json.loads((Path(tmp) / "logs" / "m3_requirements.json").read_text(encoding="utf-8"))
            self.assertFalse(result["ok"])
            self.assertEqual(len(report["checks"]), len(result["checks"]))

    def test_parquet_files(self):
        toy = _toy()
        with tempfile.TemporaryDirectory() as tmp:
            try:
                for name in ("entries", "refs", "pubs", "labels", "predictions"):
                    path = Path(tmp) / ARTIFACTS[name][0]
                    path.parent.mkdir(parents=True, exist_ok=True)
                    toy[name].to_parquet(path, index=False)
            except ImportError as err:
                self.skipTest(f"no Parquet engine: {err}")
            split = Path(tmp) / ARTIFACTS["split"][0]
            split.write_text(json.dumps(toy["split"]), encoding="utf-8")
            (Path(tmp) / ARTIFACTS["metrics"][0]).parent.mkdir(parents=True, exist_ok=True)
            (Path(tmp) / ARTIFACTS["metrics"][0]).write_text(json.dumps(toy["metrics"]), encoding="utf-8")
            data, status = load_m3_artifacts(tmp)
            self.assertEqual(set(check_requirements(data)["status"]), {"PASS"})

Opened on its own (`RUN_TEST = True`): the tests, then the state of the real files (nothing is written).

In [ ]:
if RUN_TEST:
    _result = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(AnalysisTests))
    assert _result.wasSuccessful(), "24: tests failed"
    _ = run_analysis(save=False)